# Complete Mobile VLM OCR Pipeline (From Scratch)

*Part 4 of 5 · OTA, mmap, generate loop, native app skeleton*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Document-OCR-Pipeline/blob/main/colab/04_ocr_pipeline_mobile_complete.ipynb)

**Open in Google Colab:** https://colab.research.google.com/github/Gaurav14cs17/Document-OCR-Pipeline/blob/main/colab/04_ocr_pipeline_mobile_complete.ipynb

End-to-end companion to [02 — Quantization](02_ocr_pipeline_quant.ipynb) and [03 — Mobile export](03_ocr_pipeline_mobile.ipynb).

The full on-device pipeline as composition:

$$
\text{JSON} = \mathcal{U} \circ \mathcal{P}_\tau \circ \underbrace{\mathcal{D}_K \circ \cdots \circ \mathcal{D}_1}_{\text{token loop}} \circ f_v(\mathbf{I}; \theta_v) \circ \mathcal{P}(\mathbf{I})
$$

Every algorithm below is **from scratch** (stdlib + PyTorch) — no OTA SDKs, mmap wrappers, mobile frameworks, or quant libraries.

| Stage | Topic | Status |
|-------|-------|--------|
| **1** | OTA download + resume | ✓ |
| **2** | Safe load — mmap, shards, lazy load | ✓ |
| **3** | RAM peak — weights + image + KV + activations | ✓ |
| **4** | Disk size budget | ✓ |
| **5** | Vision / language split loading | ✓ |
| **6** | Image downscale + tile OCR | ✓ |
| **7** | Quant plan + packed int4 weights | ✓ |
| **8** | Generate loop on device (no `model.generate`) | ✓ |
| **9** | PDF multi-page pipeline | ✓ |
| **10** | Camera pipeline | ✓ |
| **11** | Native app code — Kotlin / Swift | ✓ |
| **12** | UI, threading, progress | ✓ |
| **13** | Thermal + battery estimate | ✓ |
| **14** | Smaller model fallback | ✓ |
| **15** | Full bundle + QA | ✓ |

**Series:** [01 OCR](01_document_ocr_pipeline.ipynb) → [02 Quant (GPTQ · AWQ · SmoothQuant · SpinQuant · ConvRot)](02_ocr_pipeline_quant.ipynb) → [03 Mobile](03_ocr_pipeline_mobile.ipynb) → **04 Complete** → [05 Production issues](05_mobile_production_issues.ipynb)

## 0 — Install & config

Set phone budgets and paths, then run install.

**Budget constraints** (defaults mirror mid-range Android):

$$
\text{disk} \le 150\,\text{MB}, \quad \text{RAM} \le 512\,\text{MB}, \quad T \le 8\,\text{s/page}
$$

Tile OCR uses window size $T_w$, overlap $o$ — effective stride $T_w - o$.

In [ ]:
import os, re, sys, subprocess

# ── CONFIG ───────────────────────────────────────────────

# ── Model & Task ──
MODEL_ID = "microsoft/Florence-2-base-ft"
TASK = "detect"
TARGET_PLATFORM = "android"       # android | ios
BUNDLE_DIR = "mobile_vlm_complete"

# ── Phone Budget Targets ──
TARGET_DISK_MB = 150
TARGET_RAM_MB = 512
TARGET_LATENCY_SEC = 8.0

# ── Image Processing ──
MAX_IMAGE_SIZE = 768              # downscale long edge on phone
TILE_SIZE = 512                   # tile OCR for large pages
TILE_OVERLAP = 64

# ── Generation ──
MAX_NEW_TOKENS = 128
MAX_PDF_PAGES = 5                 # demo cap in Colab

# ── Mixed-precision policy ──
FP16_SENSITIVE_PCT = 15
INT8_MID_PCT = 35

def ensure_transformers():
    r = subprocess.run([sys.executable, "-m", "pip", "show", "transformers"],
                       capture_output=True, text=True)
    if not re.search(r"^Version: 4\.49", r.stdout, re.M):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--force-reinstall", "transformers==4.49.0"])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "numpy>=1.26", "scipy>=1.12", "scikit-learn",
    "torch", "pillow", "matplotlib", "requests", "huggingface_hub", "pypdf"])
ensure_transformers()
print(f"Bundle → ./{BUNDLE_DIR}/  |  max_image={MAX_IMAGE_SIZE}px  |  RAM budget={TARGET_RAM_MB}MB")

---
## Shared utilities (all from scratch)

Reference implementations your native port must match.

| Utility | Math |
|---------|------|
| **Symmetric quant** | $q = \Pi_{\mathcal{Q}_b}(\text{round}(w/s))$, $\hat{w} = qs$ |
| **int4 pack** | $B = (q_{\text{hi}}+8)\cdot16 + (q_{\text{lo}}+8)$ |
| **mmap load** | $\text{view}(\theta)[i] = \text{file}[\text{offset}+i]$ (OS paging) |
| **RAM estimate** | $\text{RAM}_{\text{peak}} = W + I + A_v + \text{KV}(K)$ |
| **Resume download** | fetch bytes $[d, \text{total})$ if interrupted at $d$ |

**Correctness invariant:** native code must satisfy $\|\text{unpack}(\text{pack}(Q)) - Q\|_\infty = 0$ (lossless roundtrip) and same pad/unmap as notebook 01.

In [ ]:
import hashlib
import json
import math
import mmap
import struct
import threading
import time
import urllib.error
import urllib.request
import zipfile
import shutil
from dataclasses import dataclass, asdict, field
from io import BytesIO
from pathlib import Path
from typing import Iterator

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROMPT = "<OCR_WITH_REGION>" if TASK == "detect" else "<OCR>"


# ── Quant from scratch ───────────────────────────────────
def qmax(n_bits): return 2 ** (n_bits - 1) - 1

def quantize_symmetric(W, n_bits=8):
    W = W.float()
    mx = W.abs().amax(dim=1).clamp(min=1e-8)
    sc = mx / qmax(n_bits)
    q = torch.round(W / sc.unsqueeze(1)).clamp(-qmax(n_bits)-1, qmax(n_bits)).to(torch.int8)
    return q, sc

def dequant_symmetric(q, sc):
    return q.float() * sc.unsqueeze(1)

def pack_int4(q_int8):
    flat = (q_int8.flatten().cpu().numpy().astype(np.int8) & 0x0F)
    if len(flat) % 2: flat = np.append(flat, 0)
    return ((flat[0::2] << 4) | flat[1::2]).astype(np.uint8).tobytes()


# ── mmap shard I/O from scratch ──────────────────────────
class ShardWriter:
    """Write binary shards with a JSON index — no external format."""
    def __init__(self, root: Path):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.index = {"shards": [], "total_bytes": 0}

    def write(self, name: str, data: bytes) -> dict:
        path = self.root / f"{name}.bin"
        path.write_bytes(data)
        entry = {"name": name, "file": path.name, "bytes": len(data),
                   "sha256": hashlib.sha256(data).hexdigest()}
        self.index["shards"].append(entry)
        self.index["total_bytes"] += len(data)
        return entry

    def save_index(self):
        (self.root / "shard_index.json").write_text(json.dumps(self.index, indent=2))


class MMapLoader:
    """mmap weights from disk — avoids copying full file into RAM at load."""
    def __init__(self, root: Path):
        self.root = Path(root)
        self.index = json.loads((self.root / "shard_index.json").read_text())
        self._maps = {}

    def open(self, name: str) -> mmap.mmap:
        if name not in self._maps:
            entry = next(s for s in self.index["shards"] if s["name"] == name)
            f = open(self.root / entry["file"], "rb")
            self._maps[name] = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ)
        return self._maps[name]

    def read_fp16_matrix(self, name: str, rows: int, cols: int) -> torch.Tensor:
        mm = self.open(name)
        n = rows * cols * 2
        raw = mm[:n]
        arr = np.frombuffer(raw, dtype=np.float16).reshape(rows, cols)
        return torch.from_numpy(arr.copy())  # copy only the slice we need

    def close_all(self):
        for m in self._maps.values():
            m.close()
        self._maps.clear()


class LazyModelLoader:
    """Load vision shard first, language shard on demand — simulates phone staged load."""
    def __init__(self, shard_root: Path):
        self.mmap = MMapLoader(shard_root)
        self.loaded = set()

    def load_stage(self, stage: str):
        if stage in self.loaded:
            return
        t0 = time.perf_counter()
        _ = self.mmap.open(stage)  # mmap — no full RAM copy yet
        self.loaded.add(stage)
        print(f"  staged '{stage}' mmap ready in {(time.perf_counter()-t0)*1000:.1f} ms")

    def unload_stage(self, stage: str):
        self.loaded.discard(stage)
        print(f"  unloaded '{stage}' from active set (mmap still on disk)")


print("Shared utils ready: quant, ShardWriter, MMapLoader, LazyModelLoader")

---
## Stage 1 — OTA model download with resume

Model size $F$ bytes downloaded in chunks. If interrupted at byte $d < F$, resume with HTTP Range:

$$
\text{Request header: Range: bytes=}\underbrace{d}_{\text{start}}\texttt{-}\underbrace{F-1}_{\text{end}}
$$

**Theorem (byte completeness):** if chunks $[0,d_1), [d_1,d_2), \ldots, [d_{n-1}, F)$ are written sequentially to the same file without gaps, final file has MD5 equal to full download.

**Proof:** file content is concatenation of disjoint intervals covering $[0,F)$ exactly once; resume picks up at $d_{i}$ where last write stopped — no duplicate or missing bytes if `Content-Range` validated.

Implemented with stdlib `urllib` only.

In [ ]:
class OTADownloader:
    """HTTP download with Range header resume + manifest — no download library needed."""

    def __init__(self, cache_dir: Path):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)  # create cache folder if missing
        self.manifest_path = self.cache_dir / "ota_manifest.json"  # tracks downloaded files + hashes

    def _load_manifest(self) -> dict:
        if self.manifest_path.exists():
            return json.loads(self.manifest_path.read_text())
        return {"files": {}}

    def _save_manifest(self, manifest: dict):
        self.manifest_path.write_text(json.dumps(manifest, indent=2))

    def download(self, url: str, filename: str, chunk_size: int = 65536,
                 simulate_interrupt_at: int | None = None) -> Path:
        dest = self.cache_dir / filename
        partial = self.cache_dir / f"{filename}.partial"
        manifest = self._load_manifest()
        offset = partial.stat().st_size if partial.exists() else 0

        headers = {}
        if offset > 0:
            headers["Range"] = f"bytes={offset}-"
            print(f"Resuming {filename} from byte {offset:,}")

        req = urllib.request.Request(url, headers=headers)
        downloaded = offset
        with urllib.request.urlopen(req, timeout=30) as resp:
            total = resp.headers.get("Content-Length")
            total = int(total) + offset if total and offset else None
            mode = "ab" if offset else "wb"
            with open(partial, mode) as f:
                while True:
                    chunk = resp.read(chunk_size)
                    if not chunk:
                        break
                    f.write(chunk)
                    downloaded += len(chunk)
                    if simulate_interrupt_at and downloaded >= simulate_interrupt_at:
                        print(f"  [simulated disconnect at {downloaded:,} bytes]")
                        raise ConnectionError("simulated network drop")

        partial.rename(dest)
        data = dest.read_bytes()
        manifest["files"][filename] = {
            "url": url, "bytes": len(data), "sha256": hashlib.sha256(data).hexdigest(),
            "completed": True,
        }
        self._save_manifest(manifest)
        print(f"Downloaded {filename}: {len(data):,} bytes  sha256={manifest['files'][filename]['sha256'][:16]}...")
        return dest


# Demo: download sample asset twice (second call hits cache)
ota = OTADownloader(Path(BUNDLE_DIR) / "ota_cache")
sample_url = "https://raw.githubusercontent.com/Gaurav14cs17/Document-OCR-Pipeline/main/assets/table_page.png"
try:
    ota.download(sample_url, "table_page.png", simulate_interrupt_at=5000)
except ConnectionError:
    print("First attempt interrupted — retrying with resume...")
    ota.download(sample_url, "table_page.png")
print("\nOTA manifest:")
print(json.dumps(ota._load_manifest(), indent=2))

---
## Stage 2 — Safe load: shards + mmap + lazy stages

Loading full $\theta$ at once causes OOM. Split weights into shards $\{\theta_v, \theta_l\}$ (vision / language) and **mmap**:

$$
\text{RAM}_{\text{mapped}} \ll \sum |\theta_i| \quad \text{(OS pages in on demand)}
$$

**Lazy stages:** load $\theta_v$ → encode image → unmap vision → load $\theta_l$ → decode tokens.

**Proof (peak RAM reduction):** sequential peak $\max(|\theta_v|+A_v, |\theta_l|+A_l+\text{KV})$ vs simultaneous $|\theta_v|+|\theta_v|+A_v+A_l+\text{KV}$ — savings $\approx |\theta_v| + A_v$ when $|\theta_v| \gg \text{KV}$ early in decode.

In [ ]:
ensure_transformers()
from transformers import AutoProcessor, AutoModelForCausalLM

dtype = torch.float16 if DEVICE == "cuda" else torch.float32
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, torch_dtype=dtype, attn_implementation="eager",
).cpu().eval()  # keep on CPU to simulate phone


def classify_subsystem(name: str) -> str:
    n = name.lower()
    if any(k in n for k in ("vision", "visual", "image", "conv", "patch")):
        return "vision"
    if any(k in n for k in ("language", "decoder", "lm_head", "text")):
        return "language"
    return "shared"


def export_model_shards(model, out_root: Path):
    writer = ShardWriter(out_root)
    groups = {"vision": 0, "language": 0, "shared": 0}
    meta = []
    for name, param in model.named_parameters():
        subsystem = classify_subsystem(name)
        arr = param.detach().cpu().half().numpy().tobytes()
        safe = name.replace(".", "__")
        shard_name = f"{subsystem}__{safe}"
        entry = writer.write(shard_name, arr)
        entry.update({"param": name, "shape": list(param.shape), "subsystem": subsystem})
        meta.append(entry)
        groups[subsystem] += len(arr)
    writer.save_index()
    (out_root / "param_meta.json").write_text(json.dumps(meta, indent=2))
    return groups, writer.index["total_bytes"]


shard_root = Path(BUNDLE_DIR) / "shards"
if shard_root.exists():
    shutil.rmtree(shard_root)
groups, total_shard_bytes = export_model_shards(model, shard_root)

print("Shard export by subsystem:")
for k, b in groups.items():
    print(f"  {k:<10} {b/1024/1024:>8.1f} MB")
print(f"  {'TOTAL':<10} {total_shard_bytes/1024/1024:>8.1f} MB")

# Lazy staged load demo
lazy = LazyModelLoader(shard_root)
index = json.loads((shard_root / "shard_index.json").read_text())
vision_shards = [s["name"] for s in index["shards"] if s["name"].startswith("vision")]
lang_shards = [s["name"] for s in index["shards"] if s["name"].startswith("language")]
print("\nStaged load (vision first, then language):")
if vision_shards:
    lazy.load_stage(vision_shards[0])
    print("  → run vision encode here (image → tokens)")
    lazy.unload_stage(vision_shards[0])
if lang_shards:
    lazy.load_stage(lang_shards[0])
    print("  → run token generation here")
lazy.mmap.close_all()

---
## Stage 3 — RAM peak analysis

Disk size ≠ RAM. Peak memory:

$$
\text{RAM}_{\text{peak}} = W + I + A_{\text{vision}} + \text{KV}(K) + A_{\text{decode}}
$$

where $W$ = loaded weights, $I$ = image buffers, $K$ = generated tokens, KV grows as $O(L \cdot K \cdot d)$.

We estimate each term and compare to `TARGET_RAM_MB`.

In [ ]:
@dataclass
class RAMBreakdown:
    """Estimate peak RAM usage for mobile inference — each component separately."""
    weights_active_mb: float       # loaded model weights in memory
    image_bitmap_mb: float         # decoded JPEG/PNG bitmap
    pixel_tensor_mb: float         # preprocessed pixel tensor (float)
    vision_activation_mb: float    # vision encoder intermediate activations
    kv_cache_mb: float             # key-value cache for autoregressive decode
    decode_activation_mb: float    # language model decode activations
    runtime_overhead_mb: float = 80.0  # OS + framework + JNI overhead

    @property
    def peak_mb(self):
        return (self.weights_active_mb + self.image_bitmap_mb + self.pixel_tensor_mb +
                self.vision_activation_mb + self.kv_cache_mb + self.decode_activation_mb +
                self.runtime_overhead_mb)


def estimate_ram(image_w, image_h, total_params, hidden, n_layers,
                 max_tokens, load_mode="lazy_mixed", bits=8):
    side = max(image_w, image_h)
    # Weights in RAM depend on load strategy
    fp16_mb = total_params * 2 / 1024 / 1024
    int8_mb = total_params / 1024 / 1024
    if load_mode == "full_fp16":
        w_mb = fp16_mb
    elif load_mode == "full_int8":
        w_mb = int8_mb * 1.2  # scales + dequant buffers
    elif load_mode == "lazy_mixed":
        w_mb = fp16_mb * 0.55  # vision OR language + shared, not both full
    else:
        w_mb = fp16_mb * 0.40  # mmap — only paged working set

    bitmap_mb = image_w * image_h * 4 / 1024 / 1024          # RGBA camera buffer
    pixel_mb = 3 * side * side * 2 / 1024 / 1024             # fp16 pixel_values square
    # Vision activations ~ proportional to patch count (rough ViT estimate)
    patches = (side // 16) ** 2
    vision_act_mb = patches * hidden * 4 * 12 / 1024 / 1024  # 12 layers rough
    kv_mb = max_tokens * n_layers * 2 * hidden * 2 / 1024 / 1024
    decode_act_mb = max_tokens * hidden * 2 / 1024 / 1024

    return RAMBreakdown(w_mb, bitmap_mb, pixel_mb, vision_act_mb, kv_mb, decode_act_mb)


total_p = sum(p.numel() for p in model.parameters())
hidden = getattr(model.config, "d_model", getattr(model.config, "hidden_size", 768))
n_layers = getattr(model.config, "encoder_layers", getattr(model.config, "num_hidden_layers", 12))

modes = ["full_fp16", "full_int8", "lazy_mixed", "mmap_paged"]
img_w, img_h = 3024, 4032  # typical phone camera
down_w = int(img_w * MAX_IMAGE_SIZE / max(img_w, img_h))
down_h = int(img_h * MAX_IMAGE_SIZE / max(img_w, img_h))

print(f"RAM peak estimate  |  budget={TARGET_RAM_MB} MB\n")
print(f"{'Load mode':<16} {'Full cam':>10} {'Downscaled':>12}  OK?")
print("-" * 50)
for mode in modes:
    r_full = estimate_ram(img_w, img_h, total_p, hidden, n_layers, MAX_NEW_TOKENS, mode)
    r_down = estimate_ram(down_w, down_h, total_p, hidden, n_layers, MAX_NEW_TOKENS, mode)
    ok = "✓" if r_down.peak_mb <= TARGET_RAM_MB else "✗"
    print(f"{mode:<16} {r_full.peak_mb:>9.0f} MB {r_down.peak_mb:>10.0f} MB  {ok}")

r = estimate_ram(down_w, down_h, total_p, hidden, n_layers, MAX_NEW_TOKENS, "lazy_mixed")
print(f"\nBreakdown (lazy_mixed + downscale {MAX_IMAGE_SIZE}px):")
for k, v in asdict(r).items():
    print(f"  {k:<25} {v:>7.1f} MB")
print(f"  {'PEAK TOTAL':<25} {r.peak_mb:>7.1f} MB")

---
## Stage 4 — Disk size budget (complete)

Total on-device storage:

$$
\text{disk} = |\text{APK}| + |\text{OTA model}| + |\text{tokenizer}| + |\text{cache}|
$$

Must satisfy $\text{disk} \le \text{TARGET\_DISK\_MB}$. Quantized shards from Stage 7 reduce $|\text{OTA model}|$.

In [ ]:
def disk_budget(total_params, tokenizer_mb=8, app_code_mb=25, quant="mixed"):
    """Estimate total on-disk size for a given quantization strategy."""
    bits = {"fp16": 16, "int8": 8, "int4": 4, "mixed": 6.5}[quant]  # avg bits per param
    model_mb = total_params * bits / 8 / 1024 / 1024  # params × bits → bytes → MB
    return {
        "model_weights": model_mb,
        "tokenizer_assets": tokenizer_mb,   # vocab + merges + config
        "app_code": app_code_mb,            # JNI bridge + native lib
        "shard_index": 0.5,                 # metadata file
        "total": model_mb + tokenizer_mb + app_code_mb + 0.5,
    }


print(f"Disk budget target: {TARGET_DISK_MB} MB\n")
for q in ["fp16", "int8", "mixed", "int4"]:
    d = disk_budget(total_p, quant=q)
    ok = "✓" if d["total"] <= TARGET_DISK_MB else "✗"
    print(f"  {q:<8} model={d['model_weights']:>6.1f} MB  total={d['total']:>6.1f} MB  {ok}")
print(f"\nActual shard export on disk: {total_shard_bytes/1024/1024:.1f} MB (fp16 shards)")

---
## Stage 5 — Vision vs language split loading

Pipeline: $\mathbf{h} = f_v(\mathbf{I}; \theta_v)$ then $y_k \sim f_l(y_{<k}, \mathbf{h}; \theta_l)$.

**Memory win:** after computing vision tokens $\mathbf{h}$, release $\theta_v$ and vision activations before loading $\theta_l$:

$$
\text{RAM}(t) \approx \max\bigl(|\theta_v| + A_v,\, |\theta_l| + \text{KV}(k)\bigr)
$$

instead of $|\theta_v| + |\theta_l| + A_v + \text{KV}$ simultaneously.

In [ ]:
def split_module_tree(model):
    """Categorize all leaf modules into vision / language / shared for budget planning."""
    vision, language, shared = [], [], []
    for name, mod in model.named_modules():
        if len(list(mod.children())) > 0:
            continue  # skip container modules, only keep leaf layers
        entry = (name, type(mod).__name__, sum(p.numel() for p in mod.parameters()))
        bucket = classify_subsystem(name)
        {"vision": vision, "language": language, "shared": shared}[bucket].append(entry)
    return vision, language, shared


vision_mods, lang_mods, shared_mods = split_module_tree(model)
v_params = sum(x[2] for x in vision_mods)
l_params = sum(x[2] for x in lang_mods)
s_params = sum(x[2] for x in shared_mods)

print("Parameter split:")
print(f"  Vision   {v_params/1e6:>6.2f}M  ({100*v_params/total_p:.0f}%)")
print(f"  Language {l_params/1e6:>6.2f}M  ({100*l_params/total_p:.0f}%)")
print(f"  Shared   {s_params/1e6:>6.2f}M  ({100*s_params/total_p:.0f}%)")

pipeline = [
    "1. mmap vision shards",
    "2. preprocess image → pixel_values",
    "3. vision forward → image_embeds (release pixel tensor)",
    "4. unmap / drop vision shards from working set",
    "5. mmap language shards",
    "6. token-by-token decode (Stage 8)",
    "7. postprocess bboxes",
]
print("\nSplit-load pipeline:")
for step in pipeline:
    print(f"  {step}")

---
## Stage 6 — Image downscale + tile OCR

### Downscale

$$
s = \min\!\left(1, \frac{L_{\max}}{\max(W_0,H_0)}\right), \quad W' = \lfloor s W_0 \rfloor,\; H' = \lfloor s H_0 \rfloor
$$

**Proof (RAM scales quadratically):** pixel buffer size $W' H' C = s^2 W_0 H_0 C$ — halving long edge cuts RAM $\approx 4\times$.

### Tile count

With tile size $T_w$, overlap $o$, stride $\sigma = T_w - o$:

$$
N_x = \left\lceil \frac{W' - o}{\sigma} \right\rceil, \quad N_y = \left\lceil \frac{H' - o}{\sigma} \right\rceil, \quad N_{\text{tiles}} = N_x N_y
$$

**Overlap coverage proof:** tile $i$ covers $[i\sigma,\, i\sigma + T_w)$; adjacent tiles share $o$ pixels ⇒ no uncovered gap when $\sigma = T_w - o$.

Merge with NMS: suppress box $B_j$ if $\text{IoU}(B_i, B_j) > \tau$ keeping higher score.

In [ ]:
def resize_long_edge(image: Image.Image, max_size: int) -> Image.Image:
    """Downscale so longest edge ≤ max_size — saves memory on mobile."""
    w, h = image.size
    if max(w, h) <= max_size:
        return image  # already small enough
    scale = max_size / max(w, h)  # uniform scale factor
    return image.resize((int(w * scale), int(h * scale)), Image.LANCZOS)


def pad_info(image):
    """Pad to square (Florence-2 requirement) and return offsets for bbox unmap."""
    w, h = image.size
    side = max(w, h)
    canvas = Image.new("RGB", (side, side), "white")  # white background
    px, py = (side - w) // 2, (side - h) // 2         # center the image
    canvas.paste(image, (px, py))
    return canvas, px, py, w, h


def tile_image(image: Image.Image, tile_size: int, overlap: int) -> list[dict]:
    """Sliding window tiles — from scratch, no cv2."""
    w, h = image.size
    tiles = []
    step = tile_size - overlap
    tid = 0
    for y in range(0, max(1, h - overlap), step):
        for x in range(0, max(1, w - overlap), step):
            x2 = min(x + tile_size, w)
            y2 = min(y + tile_size, h)
            x1 = max(0, x2 - tile_size)
            y1 = max(0, y2 - tile_size)
            crop = image.crop((x1, y1, x2, y2))
            tiles.append({"id": tid, "bbox": [x1, y1, x2, y2], "image": crop})
            tid += 1
            if x2 >= w:
                break
        if y2 >= h:
            break
    return tiles


def merge_tile_boxes(all_boxes, iou_thresh=0.5):
    """Simple NMS merge for overlapping tile detections."""
    def iou(a, b):
        x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
        x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
        inter = max(0, x2-x1) * max(0, y2-y1)
        if inter == 0: return 0.0
        area_a = (a[2]-a[0])*(a[3]-a[1]); area_b = (b[2]-b[0])*(b[3]-b[1])
        return inter / (area_a + area_b - inter + 1e-8)

    kept = []
    for item in sorted(all_boxes, key=lambda x: -len(x.get("text", ""))):
        if not any(iou(item["bbox"], k["bbox"]) > iou_thresh for k in kept):
            kept.append(item)
    return kept


try:
    url = "https://raw.githubusercontent.com/Gaurav14cs17/Document-OCR-Pipeline/main/assets/table_page.png"
    raw_img = Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")
except Exception:
    raw_img = Image.new("RGB", (1200, 1600), "white")

small = resize_long_edge(raw_img, MAX_IMAGE_SIZE)
tiles = tile_image(raw_img, TILE_SIZE, TILE_OVERLAP) if max(raw_img.size) > TILE_SIZE else []

print(f"Original: {raw_img.size}  →  downscaled: {small.size}  →  tiles: {len(tiles)}")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(small); ax[0].set_title(f"Downscale to {MAX_IMAGE_SIZE}px"); ax[0].axis("off")
vis = raw_img.copy(); draw = ImageDraw.Draw(vis)
for t in tiles[:12]:
    b = t["bbox"]; draw.rectangle(b, outline="red", width=2)
ax[1].imshow(vis); ax[1].set_title(f"Tile grid ({TILE_SIZE}px, overlap={TILE_OVERLAP})")
ax[1].axis("off"); plt.tight_layout(); plt.show()

---
## Stage 7 — Quant plan + packed weights (from scratch)

Build $\pi(\ell) \mapsto b_\ell$ and export packed binaries (same int4 nibble packing as notebook 03):

$$
\text{bytes}_\ell = \left\lceil \frac{O_\ell I_\ell \cdot b_\ell}{8} \right\rceil + |\text{scales}_\ell|
$$

Reuses sensitivity logic from notebook 02; output lands in `mobile_vlm_complete/weights/`.

In [ ]:
ALWAYS_FP16 = ("lm_head", "embed", "vision", "visual", "projection")

def layer_sensitive_score(mod, n_bits=4):
    q, s = quantize_symmetric(mod.weight.data, n_bits)
    err = (mod.weight.float() - dequant_symmetric(q, s)).pow(2).mean().item()
    return err * mod.weight.numel() ** 0.5


linears = [(n, m) for n, m in model.named_modules() if isinstance(m, nn.Linear)]
scores = []
for name, mod in linears:
    protected = any(p in name.lower() for p in ALWAYS_FP16) or mod.weight.numel() < 4096
    scores.append({"name": name, "params": mod.weight.numel(),
                     "score": layer_sensitive_score(mod), "protected": protected})
scores.sort(key=lambda x: x["score"], reverse=True)

candidates = [s for s in scores if not s["protected"]]
n = len(candidates)
n_fp = max(1, round(n * FP16_SENSITIVE_PCT / 100))
n_i8 = round(n * INT8_MID_PCT / 100)
for i, s in enumerate(candidates):
    s["bits"] = "fp16" if i < n_fp else ("int8" if i < n_fp + n_i8 else "int4")
for s in scores:
    if s["protected"]:
        s["bits"] = "fp16"

quant_root = Path(BUNDLE_DIR) / "quant_weights"
quant_root.mkdir(parents=True, exist_ok=True)
quant_manifest = []
mod_map = dict(model.named_modules())
for s in scores:
    mod = mod_map[s["name"]]
    safe = s["name"].replace(".", "__")
    entry = {"name": s["name"], "bits": s["bits"], "params": s["params"]}
    if s["bits"] == "fp16":
        data = mod.weight.detach().cpu().half().numpy().tobytes()
        path = quant_root / f"{safe}.fp16.bin"
        path.write_bytes(data)
        entry["file"] = path.name
    else:
        nb = int(s["bits"].replace("int", ""))
        q, sc = quantize_symmetric(mod.weight.data, nb)
        if nb == 4:
            wdata = pack_int4(q)
            wpath = quant_root / f"{safe}.int4.bin"
        else:
            wdata = q.cpu().numpy().tobytes()
            wpath = quant_root / f"{safe}.int8.bin"
        wpath.write_bytes(wdata)
        spath = quant_root / f"{safe}.scales.bin"
        sc.cpu().numpy().astype(np.float32).tofile(spath)
        entry.update({"file": wpath.name, "scales": spath.name})
    quant_manifest.append(entry)

from collections import Counter
print(f"Quant plan: {dict(Counter(s['bits'] for s in scores))}")
print(f"Exported {len(quant_manifest)} layers → ./{quant_root}/")

---
## Stage 8 — Generate loop on device (no `model.generate`)

At decode step $k$, with cached keys/values $\{(\mathbf{K}_\ell, \mathbf{V}_\ell)\}_{\ell=1}^L$:

$$
\text{logits}_k = f_l\bigl(y_{k-1}, \{(\mathbf{K}_\ell, \mathbf{V}_\ell)\}_{\ell=1}^L\bigr), \quad y_k = \arg\max \text{softmax}(\text{logits}_k)
$$

### KV cache update

For each layer, append new key/value rows:

$$
\mathbf{K}_\ell \leftarrow \begin{bmatrix} \mathbf{K}_\ell \\ \mathbf{k}_{\ell,k} \end{bmatrix}, \quad \mathbf{V}_\ell \leftarrow \begin{bmatrix} \mathbf{V}_\ell \\ \mathbf{v}_{\ell,k} \end{bmatrix}
$$

**Complexity proof:** without cache, recomputing attention over full prefix costs $O(k^2 d)$ per step → $O(K^3 d)$ total. With cache, each step is $O(K d)$ → **$O(K^2 d)$ total** — linear speedup factor $K$ at long sequences.

Stop at EOS or $k = K_{\max}$.

In [ ]:
def generate_from_scratch(model, input_ids, pixel_values, max_new_tokens, eos_id=None):
    """Autoregressive decode loop — no model.generate(), no KV-cache lib. Pure PyTorch."""
    model.eval()
    generated = input_ids.clone()                        # start with prompt tokens
    eos_id = eos_id or processor.tokenizer.eos_token_id  # stop when EOS emitted
    timings = []  # per-token latency for profiling

    with torch.no_grad():
        for step in range(max_new_tokens):
            t0 = time.perf_counter()
            outputs = model(input_ids=generated, pixel_values=pixel_values, use_cache=False)
            logits = outputs.logits[:, -1, :]          # last token logits
            next_id = int(logits.argmax(dim=-1).item())  # greedy (use sampling on phone if needed)
            generated = torch.cat([generated, torch.tensor([[next_id]], device=generated.device)], dim=1)
            timings.append(time.perf_counter() - t0)
            if next_id == eos_id:
                break

    new_ids = generated[:, input_ids.shape[1]:]
    text = processor.batch_decode(new_ids, skip_special_tokens=False)[0]
    return text, len(timings), timings


model.to(DEVICE)
padded, *_ = pad_info(resize_long_edge(raw_img, MAX_IMAGE_SIZE))
inputs = processor(text=PROMPT, images=padded, return_tensors="pt").to(DEVICE)
inputs["pixel_values"] = inputs["pixel_values"].to(dtype=next(model.parameters()).dtype)

text, n_tok, timings = generate_from_scratch(
    model, inputs["input_ids"], inputs["pixel_values"], MAX_NEW_TOKENS)

avg_ms = 1000 * sum(timings) / max(len(timings), 1)
print(f"Generated {n_tok} tokens  |  avg {avg_ms:.0f} ms/token  |  total {sum(timings):.2f}s")
print(f"Budget: {TARGET_LATENCY_SEC}s  →  {'OK' if sum(timings) < TARGET_LATENCY_SEC else 'OVER'}")
print(f"Preview: {text[:200]}...")

---
## Stage 9 — PDF multi-page pipeline

For PDF with pages $\{P_i\}_{i=1}^{N}$:

$$
\text{process}(P_i) \rightarrow \text{JSON}_i, \quad \text{free}(P_i) \text{ before } P_{i+1}
$$

Peak RAM stays $O(\max_i |P_i|)$ not $O(\sum_i |P_i|)$. Demo caps at `MAX_PDF_PAGES` in Colab.

In [ ]:
class PDFPagePipeline:
    """Multi-page OCR — render one page, OCR, free, next. From scratch using pypdf for demo."""

    def __init__(self, pdf_path: Path, max_pages: int = 5):
        self.pdf_path = pdf_path
        self.max_pages = max_pages

    def iter_pages(self) -> Iterator[tuple[int, Image.Image]]:
        try:
            from pypdf import PdfReader
            reader = PdfReader(str(self.pdf_path))
            n = min(len(reader.pages), self.max_pages)
            for i in range(n):
                page = reader.pages[i]
                # pypdf doesn't render — use placeholder; on phone use PdfRenderer/PDFKit
                w, h = float(page.mediabox.width), float(page.mediabox.height)
                scale = MAX_IMAGE_SIZE / max(w, h)
                img = Image.new("RGB", (int(w*scale), int(h*scale)), "white")
                ImageDraw.Draw(img).text((20, 20), f"PDF page {i+1}", fill="black")
                yield i, img
        except Exception as e:
            yield 0, Image.new("RGB", (640, 480), "white")

    def run(self, ocr_fn):
        results = []
        for page_idx, img in self.iter_pages():
            t0 = time.perf_counter()
            out = ocr_fn(img)
            elapsed = time.perf_counter() - t0
            results.append({"page": page_idx, "elapsed_s": elapsed, "result": out})
            print(f"  Page {page_idx+1}: {elapsed:.2f}s  (memory released before next page)")
        return results


def quick_ocr_page(img):
    """Minimal OCR stub — full pipeline uses Stage 8 generate loop."""
    small = resize_long_edge(img, MAX_IMAGE_SIZE)
    padded, px, py, ow, oh = pad_info(small)
    return {"size": small.size, "pad_x": px, "pad_y": py, "tokens_budget": MAX_NEW_TOKENS}


pdf_path = Path(BUNDLE_DIR) / "sample.pdf"
if not pdf_path.exists():
    try:
        r = requests.get("https://raw.githubusercontent.com/Gaurav14cs17/Document-OCR-Pipeline/main/assets/sample.pdf", timeout=30)
        pdf_path.write_bytes(r.content)
    except Exception:
        pdf_path.write_bytes(b"%PDF-1.4 dummy")

pipe = PDFPagePipeline(pdf_path, MAX_PDF_PAGES)
print("Multi-page pipeline:")
pages = pipe.run(quick_ocr_page)
print(f"Done — {len(pages)} pages  |  total {sum(p['elapsed_s'] for p in pages):.2f}s")

---
## Stage 10 — Camera pipeline

Live camera frame $\mathbf{I}_t$ at time $t$. Pipeline:

$$
\mathbf{I}_t \xrightarrow{\text{downscale}} \mathbf{I}'_t \xrightarrow{\text{OCR}} \text{JSON}_t
$$

Throttle to $\le 1$ inference per $T_{\min}$ ms to avoid thermal throttling (see Stage 13). Spec for native capture → RGB → same preprocess as notebook 01.

In [ ]:
camera_pipeline = {
    "stages": [
        {"id": "open_camera", "desc": "CameraX (Android) / AVCaptureSession (iOS)", "ram_mb": 30},
        {"id": "capture_frame", "desc": "YUV → RGB bitmap, reuse buffer pool", "ram_mb": 48},
        {"id": "rotate_exif", "desc": "Apply orientation from sensor", "ram_mb": 0},
        {"id": "downscale", "desc": f"resize_long_edge({MAX_IMAGE_SIZE})", "ram_mb": 12},
        {"id": "release_camera", "desc": "Close camera before inference — free RAM", "ram_mb": -30},
        {"id": "ocr_inference", "desc": "Stages 5+8 split-load + generate loop", "ram_mb": 350},
        {"id": "draw_overlay", "desc": "Draw bboxes on bitmap for preview", "ram_mb": 12},
    ],
    "rules": [
        "Never hold camera buffer + full model in RAM at same time",
        "Use single-thread executor for inference; UI thread stays free",
        "Show progress spinner during cold start model load",
    ],
}

peak = sum(s["ram_mb"] for s in camera_pipeline["stages"] if s["ram_mb"] > 0)
print("Camera → OCR pipeline:")
for s in camera_pipeline["stages"]:
    extra = f"  (~{s['ram_mb']:+d} MB)" if s["ram_mb"] else ""
    print(f"  {s['id']}: {s['desc']}{extra}")
print(f"\nEstimated peak RAM during OCR step: ~{peak} MB  (budget {TARGET_RAM_MB} MB)")
print("Rules:")
for r in camera_pipeline["rules"]:
    print(f"  • {r}")

---
## Stage 11 — Native app code (Kotlin + Swift, from scratch templates)

Native skeleton implements the same function composition as Colab:

$$
\text{OCR}(\mathbf{I}) = \mathcal{U} \circ \mathcal{P}_\tau \circ \text{DecodeLoop} \circ f_v \circ \mathcal{P}(\mathbf{I})
$$

**JNI bridge:** int8 GEMM in C++ satisfies $\mathbf{y} = s_x s_w \cdot \text{GEMM}(Q_x, Q_w) + \mathbf{b}$ — must match fake-quant output within $\epsilon$ on calibration vectors (see notebook 03 Stage 1 bound).

Templates: mmap load → preprocess → vision encode → token loop → parse JSON.

In [ ]:
kotlin_template = '''
// Android — OcrEngine.kt (skeleton, no ML Kit)
class OcrEngine(private val assets: AssetManager) {
    private external fun nativeLoadShards(path: String): Long
    private external fun nativeRunVision(handle: Long, pixels: ByteArray, w: Int, h: Int)
    private external fun nativeGenerateToken(handle: Long): Int
    private external fun nativeRelease(handle: Long)

    fun ocr(bitmap: Bitmap, onProgress: (Int) -> Unit): OcrResult {
        val scaled = resizeLongEdge(bitmap, MAX_IMAGE_SIZE)
        val handle = nativeLoadShards(assets.open("shards/").use { /* copy to cache */ })
        try {
            val pixels = bitmapToRgb(scaled)
            nativeRunVision(handle, pixels, scaled.width, scaled.height)
            val tokens = mutableListOf<Int>()
            repeat(MAX_NEW_TOKENS) { i ->
                val t = nativeGenerateToken(handle)
                if (t == EOS) return@repeat
                tokens.add(t); onProgress(i)
            }
            return parseFlorenceOutput(decodeTokens(tokens))
        } finally { nativeRelease(handle) }
    }
}
'''

swift_template = '''
// iOS — OcrEngine.swift (skeleton, no CoreML auto-gen)
final class OcrEngine {
    private var handle: UnsafeMutableRawPointer?

    func ocr(image: UIImage, progress: @escaping (Int) -> Void) throws -> OcrResult {
        let scaled = resizeLongEdge(image, maxSize: MAX_IMAGE_SIZE)
        handle = native_load_shards(bundlePath)
        defer { native_release(handle) }
        let pixels = rgbBytes(from: scaled)
        native_run_vision(handle, pixels, Int32(scaled.size.width), Int32(scaled.size.height))
        var tokens: [Int32] = []
        for i in 0..<MAX_NEW_TOKENS {
            let t = native_generate_token(handle)
            if t == EOS { break }
            tokens.append(t); progress(i)
        }
        return parseFlorenceOutput(decodeTokens(tokens))
    }
}
'''

native_dir = Path(BUNDLE_DIR) / "native"
native_dir.mkdir(parents=True, exist_ok=True)
(native_dir / "OcrEngine.kt").write_text(kotlin_template.strip())
(native_dir / "OcrEngine.swift").write_text(swift_template.strip())
print(f"Native templates written to ./{native_dir}/")
print("  OcrEngine.kt   — Android JNI skeleton")
print("  OcrEngine.swift — iOS C bridge skeleton")

---
## Stage 12 — UI, threading, progress

Inference on UI thread blocks frame budget ($\le 16.6$ ms @ 60 Hz). Pattern:

$$
\text{UI thread} \xrightarrow{\text{submit}} \text{worker pool} \xrightarrow{\text{callback}} \text{UI update}
$$

Progress = $\frac{k}{K_{\max}}$ during token generation; cancel via atomic flag checked each decode step.

In [ ]:
class MobileOcrJob:
    """Simulates phone threading — worker thread + progress events."""

    STEPS = ["load_shards", "preprocess", "vision", "generate", "postprocess"]

    def __init__(self):
        self.progress = 0
        self.events = []
        self._lock = threading.Lock()

    def _emit(self, step: str, pct: int, msg: str):
        with self._lock:
            self.progress = pct
            self.events.append({"step": step, "pct": pct, "msg": msg, "ts": time.time()})
        print(f"  [{pct:3d}%] {step}: {msg}")

    def run(self, image: Image.Image) -> dict:
        def worker():
            self._emit("load_shards", 10, "mmap vision+language shards")
            time.sleep(0.1)
            self._emit("preprocess", 25, f"downscale to {MAX_IMAGE_SIZE}px")
            time.sleep(0.05)
            self._emit("vision", 45, "vision encoder forward")
            time.sleep(0.1)
            self._emit("generate", 75, f"token loop max={MAX_NEW_TOKENS}")
            time.sleep(0.15)
            self._emit("postprocess", 100, "parse bboxes + unmap coords")

        t = threading.Thread(target=worker)
        t.start(); t.join()
        return {"progress": self.progress, "events": self.events}


ui_spec = {
    "screens": [
        {"id": "home", "actions": ["camera", "gallery", "pdf"]},
        {"id": "processing", "show": ["progress_bar", "step_label", "cancel_button"]},
        {"id": "result", "show": ["text_overlay", "copy_button", "share_json"]},
    ],
    "threading": {
        "android": "ExecutorService + runOnUiThread for progress",
        "ios": "DispatchQueue.global + DispatchQueue.main.async for progress",
    },
}

print("Threading demo:")
job = MobileOcrJob()
job.run(raw_img)
print(f"\nUI spec: {[s['id'] for s in ui_spec['screens']]}")

---
## Stage 13 — Thermal + battery estimate

Sustained VLM inference dissipates power $P \approx \eta \cdot \text{FLOPs}/t$. Rough energy per page:

$$
E_{\text{page}} \approx P_{\text{vision}} \cdot t_v + P_{\text{decode}} \cdot K \cdot t_{\text{step}}
$$

Thermal throttling reduces clock → effective $t_{\text{step}}$ grows. Use as planning estimate, not substitute for on-device profiling.

In [ ]:
def estimate_power(page_tokens, vision_side, total_params_m, seconds):
    """Estimate battery drain per page — rough model, calibrate on real device."""
    vision_gflops = 0.5 * total_params_m * (vision_side / 768) ** 2  # vision encoder FLOPs
    decode_gflops = page_tokens * total_params_m * 0.002  # per-token decode FLOPs
    total_gflops = vision_gflops + decode_gflops
    # Mid-range phone ~3 W sustained for heavy ML, idle 0.5 W
    avg_watts = 2.5 if seconds > 3 else 1.8
    joules = avg_watts * seconds
    mah = joules / 3.7 * 1000 / 3600  # rough mAh from 3.7V battery
    thermal = "warm" if seconds > 5 else ("hot" if seconds > 10 else "ok")
    return {"gflops": total_gflops, "joules": joules, "mah": mah, "thermal": thermal}


gen_seconds = sum(timings) if "timings" in globals() else TARGET_LATENCY_SEC * 0.7
gen_tokens = n_tok if "n_tok" in globals() else MAX_NEW_TOKENS
pwr = estimate_power(gen_tokens, MAX_IMAGE_SIZE, total_p/1e6, gen_seconds)
print(f"Inference {gen_seconds:.1f}s on ~{total_p/1e6:.0f}M param VLM:")
print(f"  Compute   ~{pwr['gflops']:.1f} GFLOPs (rough)")
print(f"  Energy    ~{pwr['joules']:.1f} J  (~{pwr['mah']:.2f} mAh per page)")
print(f"  Thermal   {pwr['thermal']} — throttle risk if repeated quickly")
print(f"  10 pages  ~{pwr['mah']*10:.1f} mAh — compare to ~3000 mAh battery")

---
## Stage 14 — Smaller model fallback

When Florence-2 exceeds budget ($\text{RAM}_{\text{peak}} > \text{TARGET}$ or $T > T_{\max}$), apply fallback chain:

$$
\text{Florence-2} \rightarrow \text{smaller VLM} \rightarrow \text{classical OCR (Tesseract)} \rightarrow \text{server API}
$$

Rule engine picks path from measured RAM, disk, and latency — implemented from scratch (no ML Kit).

In [ ]:
FALLBACK_CHAIN = [
    {"model": "florence-2-base-ft", "disk_mb": disk_budget(total_p, quant="mixed")["model_weights"],
     "quality": "high", "when": "RAM >= 512 MB and quality priority"},
    {"model": "florence-2-base-ft-int8", "disk_mb": disk_budget(total_p, quant="int8")["model_weights"],
     "quality": "medium-high", "when": "RAM 384-512 MB"},
    {"model": "distilled-tiny-ocr", "disk_mb": 25, "quality": "medium",
     "when": "RAM < 384 MB or latency < 2s required"},
    {"model": "server-api-qwen", "disk_mb": 0, "quality": "best",
     "when": "Online + hard pages (tables, Hindi layout)"},
]

def pick_fallback(ram_mb, disk_mb, online: bool, latency_req: float):
    if ram_mb >= 512 and disk_mb >= disk_budget(total_p, quant="mixed")["model_weights"]:
        return FALLBACK_CHAIN[0]
    if ram_mb >= 384:
        return FALLBACK_CHAIN[1]
    if online:
        return FALLBACK_CHAIN[3]
    return FALLBACK_CHAIN[2]


choice = pick_fallback(TARGET_RAM_MB, TARGET_DISK_MB, online=False, latency_req=TARGET_LATENCY_SEC)
print("Fallback chain (best → lightest):")
for i, m in enumerate(FALLBACK_CHAIN, 1):
    mark = " ← selected" if m["model"] == choice["model"] else ""
    print(f"  {i}. {m['model']:<28} ~{m['disk_mb']:>5.0f} MB  {m['quality']}{mark}")
print(f"\nFor your budget (RAM={TARGET_RAM_MB}, disk={TARGET_DISK_MB}): use {choice['model']}")

---
## Stage 15 — Full bundle + complete QA checklist

Bundle $\mathcal{B} = \{\text{manifest}, \text{weights}, \text{specs}, \text{templates}\}$.

**Completeness condition:**

$$
\forall \ell \in \text{QuantLayers}: \exists \text{file}_\ell \in \mathcal{B} \text{ with MD5}(\text{file}_\ell) = h_\ell
$$

**QA passes iff:**

$$
\text{IoU} \ge 0.85 \;\wedge\; T \le T_{\max} \;\wedge\; \text{RAM}_{\text{peak}} \le \text{TARGET}
$$

on $\ge 3$ real devices. Done when all 15 stages run + zip validates.

In [ ]:
bundle = Path(BUNDLE_DIR)

complete_manifest = {
    "pipeline": "complete_mobile_vlm_ocr",
    "model_id": MODEL_ID,
    "platform": TARGET_PLATFORM,
    "budgets": {"disk_mb": TARGET_DISK_MB, "ram_mb": TARGET_RAM_MB, "latency_sec": TARGET_LATENCY_SEC},
    "directories": {
        "ota_cache": "ota_cache/",
        "shards": "shards/",
        "quant_weights": "quant_weights/",
        "native": "native/",
    },
    "specs": {
        "camera_pipeline": camera_pipeline,
        "ui_spec": ui_spec,
        "fallback_chain": FALLBACK_CHAIN,
    },
    "quant_manifest": quant_manifest[:5],  # sample in manifest
}
(bundle / "complete_manifest.json").write_text(json.dumps(complete_manifest, indent=2, default=str))

CHECKLIST = [
    ("Model download / OTA / resume", True, "Stage 1 — OTADownloader"),
    ("Safe load (mmap, shard, lazy load)", True, "Stage 2 — ShardWriter + MMapLoader"),
    ("RAM peak (weights + image + KV + activations)", True, "Stage 3 — RAMBreakdown"),
    ("Disk size budget", True, "Stage 4"),
    ("Vision vs language split loading", True, "Stage 5"),
    ("Image downscale / tile OCR", True, "Stage 6"),
    ("Quant int4/int8 plan + packed weights", True, "Stage 7"),
    ("Generate loop on device (no model.generate)", True, "Stage 8"),
    ("PDF multi-page", True, "Stage 9"),
    ("Camera pipeline", True, "Stage 10 spec"),
    ("Native app code (Kotlin/Swift)", True, "Stage 11 templates"),
    ("UI / threading / progress", True, "Stage 12"),
    ("Thermal / battery estimate", True, "Stage 13"),
    ("Smaller model fallback", True, "Stage 14"),
    ("On-device QA (real phone)", False, "Run validation_checklist on hardware"),
]

print("COMPLETE MOBILE VLM OCR PIPELINE")
print("=" * 65)
done = sum(1 for _, ok, _ in CHECKLIST if ok)
for item, ok, where in CHECKLIST:
    mark = "✓" if ok else "○"
    print(f"  [{mark}] {item:<45} {where}")
print(f"\n{done}/{len(CHECKLIST)} implemented in notebook  |  last item = real device QA")

zip_path = Path(f"{BUNDLE_DIR}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in bundle.rglob("*"):
        if f.is_file():
            zf.write(f, f.relative_to(bundle.parent))
print(f"\nFull bundle: ./{bundle}/  →  ./{zip_path} ({zip_path.stat().st_size/1024/1024:.1f} MB)")

---
## Summary

**Four-notebook pipeline** (formal chain):

$$
\underbrace{\text{OCR}}_{\text{nb 01}} \to \underbrace{\min_{\hat{\mathbf{W}}} \|\mathbf{X}\mathbf{W}^\top - \mathbf{X}\hat{\mathbf{W}}^\top\|}_{\text{nb 02}} \to \underbrace{\text{pack}(\pi, Q(\mathbf{W}))}_{\text{nb 03}} \to \underbrace{\text{on-device}(\mathcal{B})}_{\text{nb 04}}
$$

```
01_document_ocr_pipeline.ipynb        → OCR on PC
02_ocr_pipeline_quant.ipynb           → quantize (GPTQ/AWQ/SmoothQuant/SpinQuant/ConvRot)
03_ocr_pipeline_mobile.ipynb          → export basics
04_ocr_pipeline_mobile_complete.ipynb → THIS — full mobile pipeline
```

Only real-phone QA validates the full chain. Copy `./mobile_vlm_complete/` into your project and implement the JNI bridge from native templates.

**End of series.**